# Multilingual Agent Call - ElevenLabs API Demo

I built the "declined card / fraud hold" support call from my **ElevenAgents financial-services concept** as real audio using the ElevenLabs API, in three languages: **English, Spanish, and German**.

**Why these three:** Spanish and English cover huge financial-services markets in the US and EU. German is deliberate, because Deutsche Telekom is a named ElevenLabs enterprise customer, so it is a realistic language to prove the product on.

This notebook runs entirely in Google Colab. Nothing to install on your computer. Run the cells from top to bottom.

> Your API key is entered at runtime and is **never saved** in this notebook or committed to GitHub.

## Step 1. Install the libraries
`elevenlabs` is the official SDK. `pydub` stitches the separate lines into one continuous call.

In [ ]:
!pip install elevenlabs pydub -q

## Step 2. Enter your ElevenLabs API key
Get a free key at elevenlabs.io (Profile -> API Keys). When you run this cell, a box appears. Paste your key and press Enter. It is held only in memory for this session.

In [ ]:
from getpass import getpass
from elevenlabs.client import ElevenLabs

api_key = getpass("Paste your ElevenLabs API key, then press Enter: ")
client = ElevenLabs(api_key=api_key)
print("Connected to ElevenLabs.")

## Step 3. Choose the two voices
One voice plays the caller, one plays the support agent. These are two stock ElevenLabs voice IDs. To use different voices, browse the Voice Library in your ElevenLabs dashboard, copy a voice ID, and paste it below.

In [ ]:
# Two distinct voices. Swap these IDs for any voice from your ElevenLabs Voice Library.
VOICE_CALLER = "21m00Tcm4TlvDq8ikWAM"  # caller
VOICE_AGENT  = "JBFqnCBsd6RMkjVDRZzb"  # support agent

# The multilingual model detects the language from the text automatically.
MODEL = "eleven_multilingual_v2"
OUTPUT_FORMAT = "mp3_44100_128"

## Step 4. The call script, in three languages
The same "declined card, fraud hold, resolved with no human handoff" conversation. Each line is tagged `caller` or `agent`, which decides the voice.

In [ ]:
call = {
    "english": [
        ("caller", "Hi, my card just got declined and I'm travelling tomorrow. I really need this sorted."),
        ("agent",  "I can help with that. I can see a fraud hold was placed after a charge in Lisbon. Was that charge you?"),
        ("caller", "Yes, that was me."),
        ("agent",  "Thank you. I've cleared the hold, your card is active again, and I've noted your travel so this won't happen tomorrow."),
    ],
    "spanish": [
        ("caller", "Hola, me acaban de rechazar la tarjeta y viajo mañana. Necesito resolverlo."),
        ("agent",  "Puedo ayudarle. Veo que se aplicó un bloqueo por fraude tras un cargo en Lisboa. ¿Fue usted ese cargo?"),
        ("caller", "Sí, fui yo."),
        ("agent",  "Gracias. He retirado el bloqueo, su tarjeta vuelve a estar activa, y he registrado su viaje para que mañana no vuelva a ocurrir."),
    ],
    "german": [
        ("caller", "Hallo, meine Karte wurde gerade abgelehnt und ich reise morgen. Ich muss das dringend klären."),
        ("agent",  "Das kläre ich gern. Ich sehe eine Betrugssperre nach einer Zahlung in Lissabon. War diese Zahlung von Ihnen?"),
        ("caller", "Ja, das war ich."),
        ("agent",  "Danke. Ich habe die Sperre aufgehoben, Ihre Karte ist wieder aktiv, und ich habe Ihre Reise vermerkt, damit es morgen nicht erneut passiert."),
    ],
}
print("Script loaded:", ", ".join(call.keys()))

## Step 5. Generate the audio
For each language, this generates every line with the right voice, then stitches the lines together with a short pause into one call. You will get three files: `call_english.mp3`, `call_spanish.mp3`, `call_german.mp3`.

In [ ]:
import io
from pydub import AudioSegment

def say(text, voice_id):
    """Generate one line and return it as an audio segment."""
    audio_iter = client.text_to_speech.convert(
        text=text,
        voice_id=voice_id,
        model_id=MODEL,
        output_format=OUTPUT_FORMAT,
    )
    audio_bytes = b"".join(audio_iter)
    return AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")

pause = AudioSegment.silent(duration=400)  # 0.4s gap between speakers

for language, lines in call.items():
    full_call = AudioSegment.silent(duration=0)
    for role, text in lines:
        voice = VOICE_CALLER if role == "caller" else VOICE_AGENT
        full_call += say(text, voice) + pause
    filename = f"call_{language}.mp3"
    full_call.export(filename, format="mp3")
    print("Saved", filename)

print("\nDone. Three calls generated.")

## Step 6. Listen, then download
Play each call to check it, then download all three to upload into your GitHub repo's `audio` folder.

In [ ]:
from IPython.display import Audio, display

for language in call.keys():
    print(language.title())
    display(Audio(f"call_{language}.mp3"))

In [ ]:
from google.colab import files
for language in call.keys():
    files.download(f"call_{language}.mp3")

## Step 7. Write down what you noticed
Before you close this, jot two or three honest, specific observations while the audio is fresh: how natural the turn-taking felt, where pronunciation or pacing was strong or weak across the three languages, anything that surprised you, and one thing it changed about how you would position the product. Those notes go in the repo README, and they are the most important part of this whole demo.